# Bocado: asistente experto en nutrición
**Proyecto Final de IA Generativa**

Bocado es un asistente que responde preguntas sobre nutrición y alimentación saludable. Usa RAG (búsqueda de información en documentos), Gemini, ChromaDB y LangGraph.

**Cómo funciona:**
1. Se leen 3 PDFs sobre nutrición.
2. Se limpian y se dividen en fragmentos pequeños.
3. Los fragmentos se convierten en vectores con Gemini Embeddings y se guardan en ChromaDB.
4. Cuando el usuario pregunta, se buscan los fragmentos más parecidos.
5. Gemini responde usando solo esos fragmentos y recuerda la conversación.

**Nota:** los 3 PDFs se han escrito para este proyecto, basándose en recomendaciones de la OMS y la EFSA. Son solo informativos y no sustituyen a un profesional sanitario.

## 1. Carga de documentos
En esta celda se cargan las librerías y la clave de API, y se leen los 3 PDFs de la carpeta `data/` página a página. A cada página se le guarda el nombre del documento y el número de página, para poder citar las fuentes en las respuestas.

In [ ]:
# importo lo que necesito
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader

# cargo la clave del archivo .env
load_dotenv(override=True)
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError(
        "No se encuentra la clave. Crea un archivo .env con la línea "
        "GEMINI_API_KEY=tu_clave (mira el README, paso 7)."
    )

# lista de mis 3 documentos: archivo -> nombre bonito
DOCS = {
    "data/Nutricion_1_Fundamentos.pdf": "Fundamentos de la Nutrición",
    "data/Nutricion_2_Alimentacion_Saludable.pdf": "Alimentación Saludable y Dieta Mediterránea",
    "data/Nutricion_3_Habitos_y_Mitos.pdf": "Hábitos, Menús y Mitos",
}

# aquí guardo todas las páginas
paginas = []

# recorro cada PDF y lo leo
for fichero, nombre in DOCS.items():
    for i, pagina in enumerate(PyPDFLoader(fichero).load()):
        pagina.metadata["documento"] = nombre   # guardo de qué documento viene
        pagina.metadata["pagina"] = i + 1       # guardo el número de página
        paginas.append(pagina)

print("Páginas cargadas:", len(paginas))

/var/folders/dc/d0drjtf15g38vln560vcn13r0000gn/T/ipykernel_9721/2788813013.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Páginas cargadas: 23


## 2. Limpieza y división del texto
Primero se limpia el texto, se quitan los números sueltos, los espacios repetidos y las palabras cortadas por guion. Después se divide en fragmentos de 800 caracteres con 120 de solape.

Los fragmentos pequeños permiten encontrar justo la parte que responde a la pregunta. El solape evita que una idea quede cortada entre dos fragmentos.

In [2]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

# función para limpiar el texto de cada página
def limpiar(texto):
    texto = re.sub(r"-\n", "", texto)                       # junto palabras cortadas con guion
    texto = re.sub(r"^\s*\d+\s*$", "", texto, flags=re.M)   # quito líneas que son solo un número
    texto = re.sub(r"[ \t]+", " ", texto)                   # quito espacios repetidos
    texto = re.sub(r"\n{3,}", "\n\n", texto)                # quito saltos de línea de más
    return texto.strip()

# limpio y me quedo solo con páginas que tienen texto de verdad
paginas_ok = []
for p in paginas:
    p.page_content = limpiar(p.page_content)
    if len(p.page_content) >= 80:
        paginas_ok.append(p)

print("Páginas útiles:", len(paginas_ok), "de", len(paginas))

# troceo en fragmentos de 800 caracteres (con 120 de solape)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(paginas_ok)

# le pongo un número a cada fragmento
for n, c in enumerate(chunks):
    c.metadata["chunk_id"] = n

print("Fragmentos (chunks):", len(chunks))

Páginas útiles: 23 de 23
Fragmentos (chunks): 73


## 3. Base de datos vectorial (ChromaDB)
Cada fragmento se convierte en una lista de números (un *embedding*) con Gemini y se guarda en ChromaDB. Así se puede buscar por significado y no solo por palabras exactas.

Al final de la celda hay 3 pruebas de búsqueda para comprobar que se encuentran los fragmentos correctos.

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# modelo que convierte texto en números (embeddings)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", google_api_key=API_KEY)

# creo la base de datos vectorial (nombre nuevo)
vectorstore = Chroma(
    collection_name="nutricion",
    embedding_function=embeddings,
    persist_directory="chroma_db",
)

# si ya tenía datos, los borro para no duplicar
if vectorstore._collection.count() > 0:
    vectorstore.delete_collection()
    vectorstore = Chroma(
        collection_name="nutricion",
        embedding_function=embeddings,
        persist_directory="chroma_db",
    )

# guardo todos los fragmentos
vectorstore.add_documents(chunks)
print("Chunks indexados:", vectorstore._collection.count())

# pruebas: busco 3 preguntas y veo qué fragmentos encuentra
for q in ["¿Cuántas calorías tiene un gramo de grasa?",
          "¿Qué es la dieta mediterránea?",
          "¿Es verdad que los hidratos engordan?"]:
    print("\nPREGUNTA:", q)
    for doc, score in vectorstore.similarity_search_with_score(q, k=3):
        print(f"  [{doc.metadata['documento']}, pág. {doc.metadata['pagina']}] (distancia {score:.3f})")

Chunks indexados: 73

PREGUNTA: ¿Cuántas calorías tiene un gramo de grasa?
  [Fundamentos de la Nutrición, pág. 2] (distancia 0.514)
  [Fundamentos de la Nutrición, pág. 2] (distancia 0.636)
  [Fundamentos de la Nutrición, pág. 2] (distancia 0.647)

PREGUNTA: ¿Qué es la dieta mediterránea?
  [Alimentación Saludable y Dieta Mediterránea, pág. 3] (distancia 0.489)
  [Alimentación Saludable y Dieta Mediterránea, pág. 3] (distancia 0.531)
  [Alimentación Saludable y Dieta Mediterránea, pág. 4] (distancia 0.547)

PREGUNTA: ¿Es verdad que los hidratos engordan?
  [Hábitos, Menús y Mitos, pág. 8] (distancia 0.544)
  [Fundamentos de la Nutrición, pág. 3] (distancia 0.563)
  [Fundamentos de la Nutrición, pág. 3] (distancia 0.613)


## 4. Creación del agente con LangGraph
En esta celda se construye el agente. Funciona en 3 pasos:
1. **Reformular:** si la pregunta depende de la anterior (por ejemplo, "¿y la grasa?"), se reescribe de forma completa.
2. **Recuperar:** se buscan en ChromaDB los 5 fragmentos más parecidos.
3. **Generar:** Gemini responde usando esos fragmentos.

También se define el **prompt del sistema** (Bocado). Indica a Gemini que use solo los documentos, que no invente datos, que no dé consejos médicos y que cite las fuentes.

La **memoria** se guarda por conversación mediante un identificador (`thread_id`).

In [4]:
# importo lo que necesito para el agente
from typing import TypedDict, Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

# creo el modelo de lenguaje (Gemini)
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=API_KEY)

# creo el buscador: devuelve los 5 fragmentos más parecidos a la pregunta
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# este es el prompt del sistema: le dice a Gemini cómo debe comportarse
SYSTEM_PROMPT = """Eres Bocado, un asistente experto en nutrición y alimentación saludable.

OBJETIVO
Explicar de forma clara y sencilla conceptos de nutrición usando SOLO el contexto que te doy.

REGLAS
1. Responde únicamente con la información de <contexto>. No inventes datos ni cifras.
2. Si el contexto no tiene la respuesta, di exactamente: "No tengo información suficiente en mis documentos para responder a eso."
3. Si la pregunta no trata de nutrición o alimentación, di que solo puedes ayudar con esos temas.
4. Empieza directamente con la respuesta. No saludes.
5. Usa un lenguaje sencillo y respuestas cortas (máximo 6 frases). Usa listas solo si ayudan.
6. No des diagnósticos, ni dietas personalizadas, ni dosis de suplementos. Si la persona tiene una enfermedad, está embarazada o pide algo médico, recomienda consultar a un médico o dietista-nutricionista.
7. Si has respondido con el contexto, termina con una línea: "Fuentes: [Documento, pág. X]". Si no has usado el contexto (por ejemplo, si rechazas la pregunta), no pongas fuentes.

<contexto>
{contexto}
</contexto>"""

# el "estado" es la información que viaja entre los pasos del agente
class Estado(TypedDict):
    messages: Annotated[list, add_messages]   # toda la conversación
    pregunta_autonoma: str                    # la pregunta ya reescrita y completa
    contexto: str                             # los fragmentos encontrados
    fuentes: list                             # de dónde salen los fragmentos

# Gemini 3.5 a veces devuelve el texto en bloques; esta función lo convierte en texto normal
def extraer_texto(contenido):
    if isinstance(contenido, str):
        return contenido
    partes = []
    for bloque in contenido:
        if isinstance(bloque, dict) and "text" in bloque:
            partes.append(bloque["text"])
        elif isinstance(bloque, str):
            partes.append(bloque)
    return "".join(partes)

# PASO 1: reescribo la pregunta para que se entienda sola (usa la memoria)
def nodo_reformular(estado: Estado):
    mensajes = estado["messages"]
    pregunta = mensajes[-1].content
    # si es el primer mensaje, no hay nada que reescribir
    if len(mensajes) == 1:
        return {"pregunta_autonoma": pregunta}
    # si hay historial, le pido a Gemini que complete la pregunta
    historial = "\n".join(
        f"{'Usuario' if isinstance(m, HumanMessage) else 'Asistente'}: {extraer_texto(m.content)}"
        for m in mensajes[:-1]
    )
    instruccion = (
        "Reescribe la última pregunta del usuario para que se entienda sin la conversación. "
        "Devuelve SOLO la pregunta reescrita, sin explicaciones.\n\n"
        f"Conversación:\n{historial}\n\nÚltima pregunta: {pregunta}"
    )
    respuesta = llm.invoke(instruccion)
    return {"pregunta_autonoma": extraer_texto(respuesta.content).strip()}

# PASO 2: busco en la base de datos los fragmentos relacionados
def nodo_recuperar(estado: Estado):
    docs = retriever.invoke(estado["pregunta_autonoma"])
    contexto = "\n\n".join(
        f"[{d.metadata['documento']}, pág. {d.metadata['pagina']}]\n{d.page_content}"
        for d in docs
    )
    fuentes = sorted({f"{d.metadata['documento']}, pág. {d.metadata['pagina']}" for d in docs})
    return {"contexto": contexto, "fuentes": fuentes}

# PASO 3: Gemini genera la respuesta usando el contexto
def nodo_generar(estado: Estado):
    sistema = SystemMessage(content=SYSTEM_PROMPT.format(contexto=estado["contexto"]))
    respuesta = llm.invoke([sistema] + estado["messages"])
    texto = extraer_texto(respuesta.content)
    return {"messages": [AIMessage(content=texto)]}

# construyo el grafo: reformular -> recuperar -> generar
grafo = StateGraph(Estado)
grafo.add_node("reformular", nodo_reformular)
grafo.add_node("recuperar", nodo_recuperar)
grafo.add_node("generar", nodo_generar)
grafo.add_edge(START, "reformular")
grafo.add_edge("reformular", "recuperar")
grafo.add_edge("recuperar", "generar")
grafo.add_edge("generar", END)

# la memoria guarda la conversación de cada hilo (thread_id)
memoria = InMemorySaver()
app = grafo.compile(checkpointer=memoria)

# función para hacer una pregunta de forma fácil
def preguntar(texto, thread_id="hilo1"):
    config = {"configurable": {"thread_id": thread_id}}
    resultado = app.invoke({"messages": [HumanMessage(content=texto)]}, config=config)
    respuesta = resultado["messages"][-1].content
    return respuesta, resultado["pregunta_autonoma"], resultado["fuentes"]

print("Agente creado correctamente")

Agente creado correctamente


## 5. Prueba de memoria
Esta celda comprueba que el agente recuerda la conversación.
- **Conversación A:** se pregunta si se recomiendan las legumbres y después "¿Y cuántas veces por semana?". La segunda pregunta solo se entiende con la primera.
- **Conversación B:** se hace la misma segunda pregunta sin historial. Aquí el agente no sabe de qué se habla.

Así se demuestra que la memoria funciona y que cada conversación es independiente.

In [5]:
# HILO A: dos preguntas seguidas en la misma conversación
print("=== CONVERSACIÓN A ===")
for q in ["¿Se recomienda comer legumbres?",
          "¿Y cuántas veces por semana?"]:
    resp, reformulada, fuentes = preguntar(q, thread_id="memoria_A")
    print("\nUsuario:", q)
    print("Pregunta entendida:", reformulada)
    print("Asistente:", resp)

# HILO B: la misma segunda pregunta, pero sin historial
print("\n\n=== CONVERSACIÓN B (sin memoria previa) ===")
q = "¿Y cuántas veces por semana?"
resp, reformulada, fuentes = preguntar(q, thread_id="memoria_B")
print("\nUsuario:", q)
print("Pregunta entendida:", reformulada)
print("Asistente:", resp)

=== CONVERSACIÓN A ===

Usuario: ¿Se recomienda comer legumbres?
Pregunta entendida: ¿Se recomienda comer legumbres?
Asistente: Sí, se recomienda tomar legumbres varias veces por semana (de 3 a 4 días a la semana). Son una fuente excelente de proteína vegetal, fibra, hierro y folato, y se pueden consumir solas, en guisos, ensaladas, cremas o hummus.

Fuentes: [Alimentación Saludable y Dieta Mediterránea, pág. 2; Hábitos, Menús y Mitos, pág. 2]

Usuario: ¿Y cuántas veces por semana?
Pregunta entendida: ¿Cuántas veces por semana se recomienda comer legumbres?
Asistente: Se recomienda consumir legumbres de 3 a 4 días a la semana. Puedes elegirlas como una de las fuentes de proteína para repartir en las comidas principales de la semana.

Fuentes: [Hábitos, Menús y Mitos, pág. 2; Alimentación Saludable y Dieta Mediterránea, pág. 4]


=== CONVERSACIÓN B (sin memoria previa) ===

Usuario: ¿Y cuántas veces por semana?
Pregunta entendida: ¿Y cuántas veces por semana?
Asistente: No tengo informa

## 6. Cinco ejemplos de preguntas
Cinco preguntas que muestran lo que sabe hacer el asistente:
1. **Dato concreto:** frutas y verduras al día según la OMS.
2. **Concepto:** qué es la dieta mediterránea.
3. **Mito:** si los hidratos de carbono engordan.
4. **Memoria:** pregunta de seguimiento sobre las legumbres.
5. **Fuera de tema:** pregunta de fútbol, que el asistente debe rechazar.

Para cada ejemplo se muestra la pregunta, cómo la entiende el agente, la respuesta y los fragmentos utilizados.

In [6]:
# lista de ejemplos: (pregunta, hilo, qué demuestra)
ejemplos = [
    ("¿Cuántas frutas y verduras recomienda la OMS al día?", "ej1",
     "Dato concreto sacado de un documento"),
    ("¿Qué es la dieta mediterránea?", "ej2",
     "Explicación de un concepto"),
    ("¿Es verdad que los hidratos de carbono engordan?", "ej3",
     "Desmiente un mito (documento 3)"),
    ("¿Y las legumbres, cuántas veces por semana?", "ej2",
     "MEMORIA: sigue hablando de la dieta mediterránea (mismo hilo que el ejemplo 2)"),
    ("¿Quién ganó el último mundial de fútbol?", "ej5",
     "Pregunta fuera de tema: debe negarse a responder"),
]

# recorro los ejemplos y muestro todo
for n, (pregunta, hilo, que_demuestra) in enumerate(ejemplos, start=1):
    resp, reformulada, fuentes = preguntar(pregunta, thread_id=hilo)
    print(f"\n{'='*60}")
    print(f"EJEMPLO {n}: {que_demuestra}")
    print(f"Pregunta: {pregunta}")
    print(f"Pregunta entendida: {reformulada}")
    print(f"Respuesta:\n{resp}")
    print(f"Fragmentos usados: {fuentes}")


EJEMPLO 1: Dato concreto sacado de un documento
Pregunta: ¿Cuántas frutas y verduras recomienda la OMS al día?
Pregunta entendida: ¿Cuántas frutas y verduras recomienda la OMS al día?
Respuesta:
La OMS recomienda consumir al menos 400 gramos de frutas y verduras al día, lo que equivale a cinco raciones, sin contar patatas ni otros tubérculos con almidón.

Fuentes: [Alimentación Saludable y Dieta Mediterránea, pág. 1]
Fragmentos usados: ['Alimentación Saludable y Dieta Mediterránea, pág. 1', 'Alimentación Saludable y Dieta Mediterránea, pág. 4', 'Alimentación Saludable y Dieta Mediterránea, pág. 7', 'Fundamentos de la Nutrición, pág. 8']

EJEMPLO 2: Explicación de un concepto
Pregunta: ¿Qué es la dieta mediterránea?
Pregunta entendida: ¿Qué es la dieta mediterránea?
Respuesta:
La dieta mediterránea es el patrón de alimentación tradicional de los países de la cuenca mediterránea, como España, Italia, Grecia y Marruecos. Incluye una forma de cocinar, compartir la mesa, respetar la estaci

## 7. Chat con Bocado
Chat de texto para probar el asistente. Se escribe la pregunta y se pulsa Enter. El agente recuerda lo hablado durante la sesión. Para terminar, se escribe `salir`.

In [7]:
# chat de texto: el agente recuerda todo lo que hablamos en esta sesión
def chat():
    print("Bocado: Hola, pregúntame sobre nutrición. Escribe 'salir' para terminar.")
    while True:
        texto = input("\nTú: ")
        # si escribes salir, se acaba el chat
        if texto.strip().lower() in ["salir", "exit", "quit"]:
            print("Bocado: ¡Hasta pronto!")
            break
        # si no escribes nada, vuelvo a preguntar
        if not texto.strip():
            continue
        resp, _, _ = preguntar(texto, thread_id="chat_final")
        print("\nBocado:", resp)

chat()

Bocado: Hola, pregúntame sobre nutrición. Escribe 'salir' para terminar.

Bocado: Aquí tienes algunas opciones de desayunos saludables:
* Yogur natural con avena y fruta troceada, acompañado de un café o té sin azúcar.
* Tostada de pan integral con aceite de oliva y tomate, y una pieza de fruta.
* Porridge de avena con leche y fruta.
* Tostada de aguacate o yogur con frutos secos.
* Huevos revueltos con pan integral.

Fuentes: [Hábitos, Menús y Mitos, pág. 3]

Bocado: Para la merienda puedes tomar:
* Un puñado de nueces o una fruta.
* Un yogur natural.
* Una tostada con queso fresco.

Fuentes: [Hábitos, Menús y Mitos, pág. 3]

Bocado: No tengo información suficiente en mis documentos para responder a eso.
Bocado: ¡Hasta pronto!


## Conclusiones y limitaciones
**Qué se ha conseguido:**
- Un asistente que responde solo con información de sus documentos y cita las fuentes.
- Memoria de conversación con LangGraph.
- Un prompt del sistema pensado para evitar invenciones y consejos médicos.

**Limitaciones:**
- Solo conoce el contenido de los 3 PDFs.
- La memoria se pierde al reiniciar el notebook.
- No sustituye a un médico ni a un dietista-nutricionista.

**Posibles mejoras:**
- Añadir más documentos.
- Guardar la memoria en una base de datos.
- Crear una interfaz web con Streamlit.